# Deploy the Container Dashboard with Helm

Run this after `provision-cluster.ipynb` has built the cluster (RKE2 + NFS + Helm).

1. Reconnects to the running slice.
2. Uploads the Helm chart (`../charts/dashboard`) to node1.
3. Runs `helm upgrade --install`, which installs the first time and upgrades after that.
4. Checks the pods and calls the app through the ingress.
5. Prints an SSH tunnel command so you can open the dashboard in your own browser.

The images come from GHCR (`ghcr.io/ethancharest/csc478-frontend` and `csc478-backend`), built by the GitHub Actions workflow.

## 1. Setup

In [1]:
import time
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

SLICE_PREFIX = "echarest-"
SITE = ""                      # leave empty to find the running slice automatically

RELEASE   = "dashboard"
NAMESPACE = "csc478"
CHART_DIR = Path("../charts/dashboard").resolve()
REMOTE_CHARTS = "/home/ubuntu/charts"

# Extra helm flags, for example: "--set backend.replicas=5" or "--set image.tag=sha-515de19"
HELM_EXTRA = ""

if not (CHART_DIR / "Chart.yaml").exists():
    raise FileNotFoundError(f"{CHART_DIR} has no Chart.yaml. Is this notebook in notebooks/?")
print("Chart:", CHART_DIR)

User: EC1053226@wcupa.edu bastion key is valid!
Configuration is valid
User: EC1053226@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!
Chart: /home/fabric/projects/csc478-project/charts/dashboard


## 2. Reconnect to the cluster

In [2]:
if SITE:
    slice_obj = fablib.get_slice(name=SLICE_PREFIX + SITE)
else:
    mine = [s for s in fablib.get_slices() if s.get_name().startswith(SLICE_PREFIX)]
    if not mine:
        raise RuntimeError("No running slice found. Run provision-cluster.ipynb first.")
    if len(mine) > 1:
        raise RuntimeError("More than one slice found; set SITE: " + ", ".join(s.get_name() for s in mine))
    slice_obj = mine[0]

node1 = slice_obj.get_node("node1")
print("Using slice", slice_obj.get_name(), "- node1", node1.get_management_ip())


def run(cmd, show=True):
    """Run a command on node1 and print its output."""
    stdout, stderr = node1.execute(cmd, quiet=True)
    if show:
        print(f"$ {cmd}")
        print(stdout or stderr)
    return stdout


run("kubectl get nodes")
run("helm version --short")

Using slice echarest-GPN - node1 2610:e0:a04c:fab2:f816:3eff:fec0:b2fb
$ kubectl get nodes
NAME    STATUS   ROLES                AGE     VERSION
node1   Ready    control-plane,etcd   5m41s   v1.36.5+rke2r1
node2   Ready    <none>               3m5s    v1.36.5+rke2r1
node3   Ready    <none>               3m5s    v1.36.5+rke2r1

$ helm version --short
v3.22.1+gd6c1a0e



'v3.22.1+gd6c1a0e\n'

## 3. Upload the chart and install or upgrade

In [3]:
run(f"rm -rf {REMOTE_CHARTS} && mkdir -p {REMOTE_CHARTS}", show=False)
node1.upload_directory(str(CHART_DIR), REMOTE_CHARTS)
remote_chart = run(f"dirname $(find {REMOTE_CHARTS} -name Chart.yaml | head -1)", show=False).strip()
print("Chart uploaded to", remote_chart)

run(f"helm lint {remote_chart}")

run(f"helm upgrade --install {RELEASE} {remote_chart} "
    f"--namespace {NAMESPACE} --create-namespace "
    f"--wait --timeout 10m {HELM_EXTRA}")

Chart uploaded to /home/ubuntu/charts/dashboard
$ helm lint /home/ubuntu/charts/dashboard
==> Linting /home/ubuntu/charts/dashboard
[INFO] Chart.yaml: icon is recommended

1 chart(s) linted, 0 chart(s) failed

$ helm upgrade --install dashboard /home/ubuntu/charts/dashboard --namespace csc478 --create-namespace --wait --timeout 10m 
Release "dashboard" does not exist. Installing it now.
NAME: dashboard
LAST DEPLOYED: Wed Oct 21 19:16:46 2026
NAMESPACE: csc478
STATUS: deployed
REVISION: 1
TEST SUITE: None
NOTES:
Container Dashboard installed as release "dashboard" in namespace "csc478".

Watch it come up:
  kubectl -n csc478 get pods -o wide -w

Test from node1 through the ingress:
  curl -s http://192.168.1.1/api/health

Scale the backend:
  helm upgrade dashboard <chart> -n csc478 --reuse-values --set backend.replicas=5



'Release "dashboard" does not exist. Installing it now.\nNAME: dashboard\nLAST DEPLOYED: Wed Oct 21 19:16:46 2026\nNAMESPACE: csc478\nSTATUS: deployed\nREVISION: 1\nTEST SUITE: None\nNOTES:\nContainer Dashboard installed as release "dashboard" in namespace "csc478".\n\nWatch it come up:\n  kubectl -n csc478 get pods -o wide -w\n\nTest from node1 through the ingress:\n  curl -s http://192.168.1.1/api/health\n\nScale the backend:\n  helm upgrade dashboard <chart> -n csc478 --reuse-values --set backend.replicas=5\n'

## 4. Check the deployment

The backend has several replicas, so `/api/data` should report different pod names across calls.

In [4]:
run(f"kubectl -n {NAMESPACE} get pods -o wide")
run(f"kubectl -n {NAMESPACE} get svc,ingress,pvc")
run(f"helm -n {NAMESPACE} list")

print("==== through the ingress on node1 ====")
run("curl -s http://192.168.1.1/api/health")
for _ in range(5):
    out = run("curl -s http://192.168.1.1/api/data", show=False)
    print(out[out.find('"kubernetes"'):].strip()[:120])

$ kubectl -n csc478 get pods -o wide
NAME                                  READY   STATUS    RESTARTS   AGE   IP           NODE    NOMINATED NODE   READINESS GATES
dashboard-backend-596db96f6d-c6jrt    1/1     Running   0          17s   10.42.0.3    node1   <none>           <none>
dashboard-backend-596db96f6d-q9rb9    1/1     Running   0          17s   10.42.1.8    node3   <none>           <none>
dashboard-backend-596db96f6d-wtp9w    1/1     Running   0          17s   10.42.2.13   node2   <none>           <none>
dashboard-frontend-546978c755-6xxh8   1/1     Running   0          17s   10.42.2.12   node2   <none>           <none>
dashboard-frontend-546978c755-s2zfp   1/1     Running   0          17s   10.42.1.7    node3   <none>           <none>
dashboard-postgres-0                  1/1     Running   0          17s   10.42.2.14   node2   <none>           <none>

$ kubectl -n csc478 get svc,ingress,pvc
NAME               TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE
service/bac

## 5. Open the dashboard in your browser

The cluster is only reachable through the FABRIC bastion, so we use an SSH tunnel. Port 4000 is already published by the course's `docker-compose.yml`, so the tunnel goes from port 4000 in the container to port 80 on node1.

1. Run the cell below and copy the command it prints.
2. Open a **Jupyter terminal** (File → New → Terminal) and paste it. It will look like it hangs; that means the tunnel is open.
3. On your computer, open **http://localhost:4000**.

Press Ctrl+C in that terminal to close the tunnel.

In [5]:
tunnel = node1.get_ssh_command().replace(
    "ssh ", "ssh -N -o ExitOnForwardFailure=yes -L 0.0.0.0:4000:192.168.1.1:80 ", 1)
print(tunnel)

ssh -N -o ExitOnForwardFailure=yes -L 0.0.0.0:4000:192.168.1.1:80 -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fec0:b2fb


## 6. Experiments

Each of these demonstrates a goal from the project report. Run one at a time.

In [6]:
# Scaling: 3 -> 5 backend replicas, then watch them spread across nodes
run(f"helm upgrade {RELEASE} {remote_chart} -n {NAMESPACE} --reuse-values --set backend.replicas=5 --wait")
run(f"kubectl -n {NAMESPACE} get pods -o wide -l app.kubernetes.io/component=backend")

$ helm upgrade dashboard /home/ubuntu/charts/dashboard -n csc478 --reuse-values --set backend.replicas=5 --wait
Release "dashboard" has been upgraded. Happy Helming!
NAME: dashboard
LAST DEPLOYED: Wed Oct 21 19:19:07 2026
NAMESPACE: csc478
STATUS: deployed
REVISION: 2
TEST SUITE: None
NOTES:
Container Dashboard installed as release "dashboard" in namespace "csc478".

Watch it come up:
  kubectl -n csc478 get pods -o wide -w

Test from node1 through the ingress:
  curl -s http://192.168.1.1/api/health

Scale the backend:
  helm upgrade dashboard <chart> -n csc478 --reuse-values --set backend.replicas=5

$ kubectl -n csc478 get pods -o wide -l app.kubernetes.io/component=backend
NAME                                 READY   STATUS    RESTARTS   AGE     IP           NODE    NOMINATED NODE   READINESS GATES
dashboard-backend-596db96f6d-c6jrt   1/1     Running   0          2m30s   10.42.0.3    node1   <none>           <none>
dashboard-backend-596db96f6d-kjmb6   1/1     Running   0          8

'NAME                                 READY   STATUS    RESTARTS   AGE     IP           NODE    NOMINATED NODE   READINESS GATES\ndashboard-backend-596db96f6d-c6jrt   1/1     Running   0          2m30s   10.42.0.3    node1   <none>           <none>\ndashboard-backend-596db96f6d-kjmb6   1/1     Running   0          8s      10.42.1.9    node3   <none>           <none>\ndashboard-backend-596db96f6d-pj7x8   1/1     Running   0          8s      10.42.2.15   node2   <none>           <none>\ndashboard-backend-596db96f6d-q9rb9   1/1     Running   0          2m30s   10.42.1.8    node3   <none>           <none>\ndashboard-backend-596db96f6d-wtp9w   1/1     Running   0          2m30s   10.42.2.13   node2   <none>           <none>\n'

In [7]:
# Rollback: undo the last upgrade and show the release history
run(f"helm -n {NAMESPACE} history {RELEASE}")
run(f"helm -n {NAMESPACE} rollback {RELEASE} --wait")
run(f"kubectl -n {NAMESPACE} get pods -l app.kubernetes.io/component=backend")

$ helm -n csc478 history dashboard
REVISION	UPDATED                 	STATUS    	CHART          	APP VERSION	DESCRIPTION     
1       	Wed Oct 21 19:16:46 2026	superseded	dashboard-0.1.0	1.0        	Install complete
2       	Wed Oct 21 19:19:07 2026	deployed  	dashboard-0.1.0	1.0        	Upgrade complete

$ helm -n csc478 rollback dashboard --wait
Rollback was a success! Happy Helming!

$ kubectl -n csc478 get pods -l app.kubernetes.io/component=backend
NAME                                 READY   STATUS        RESTARTS   AGE
dashboard-backend-596db96f6d-c6jrt   1/1     Running       0          3m26s
dashboard-backend-596db96f6d-kjmb6   1/1     Terminating   0          64s
dashboard-backend-596db96f6d-pj7x8   0/1     Completed     0          64s
dashboard-backend-596db96f6d-q9rb9   1/1     Running       0          3m26s
dashboard-backend-596db96f6d-wtp9w   1/1     Running       0          3m26s



'NAME                                 READY   STATUS        RESTARTS   AGE\ndashboard-backend-596db96f6d-c6jrt   1/1     Running       0          3m26s\ndashboard-backend-596db96f6d-kjmb6   1/1     Terminating   0          64s\ndashboard-backend-596db96f6d-pj7x8   0/1     Completed     0          64s\ndashboard-backend-596db96f6d-q9rb9   1/1     Running       0          3m26s\ndashboard-backend-596db96f6d-wtp9w   1/1     Running       0          3m26s\n'

In [8]:
# Self-healing: delete one backend pod; Kubernetes replaces it
pod = run(f"kubectl -n {NAMESPACE} get pods -l app.kubernetes.io/component=backend -o name | head -1", show=False).strip()
run(f"kubectl -n {NAMESPACE} delete {pod}")
time.sleep(10)
run(f"kubectl -n {NAMESPACE} get pods -l app.kubernetes.io/component=backend")

$ kubectl -n csc478 delete pod/dashboard-backend-596db96f6d-c6jrt
pod "dashboard-backend-596db96f6d-c6jrt" deleted from csc478 namespace

$ kubectl -n csc478 get pods -l app.kubernetes.io/component=backend
NAME                                 READY   STATUS    RESTARTS   AGE
dashboard-backend-596db96f6d-gl56t   1/1     Running   0          11s
dashboard-backend-596db96f6d-q9rb9   1/1     Running   0          4m1s
dashboard-backend-596db96f6d-wtp9w   1/1     Running   0          4m1s



'NAME                                 READY   STATUS    RESTARTS   AGE\ndashboard-backend-596db96f6d-gl56t   1/1     Running   0          11s\ndashboard-backend-596db96f6d-q9rb9   1/1     Running   0          4m1s\ndashboard-backend-596db96f6d-wtp9w   1/1     Running   0          4m1s\n'

In [9]:
# Persistence: restart the database pod; log entries survive because they live on NFS
run(f"kubectl -n {NAMESPACE} delete pod {RELEASE}-postgres-0")
run(f"kubectl -n {NAMESPACE} wait --for=condition=Ready pod/{RELEASE}-postgres-0 --timeout=300s")
run("curl -s http://192.168.1.1/api/entries")

$ kubectl -n csc478 delete pod dashboard-postgres-0
pod "dashboard-postgres-0" deleted from csc478 namespace

$ kubectl -n csc478 wait --for=condition=Ready pod/dashboard-postgres-0 --timeout=300s
pod/dashboard-postgres-0 condition met

$ curl -s http://192.168.1.1/api/entries
[{"content":"This is a persistent log. Add entries using the form below - they are stored in PostgreSQL and survive container restarts.","created_at":"2026-10-21T19:17:00.045585Z","id":1}]



'[{"content":"This is a persistent log. Add entries using the form below - they are stored in PostgreSQL and survive container restarts.","created_at":"2026-10-21T19:17:00.045585Z","id":1}]\n'

## 7. Uninstall

Removes everything the chart created **except** the database Secret (kept on purpose) and the data on NFS. Reinstalling picks the old data back up.

In [11]:
UNINSTALL = True   # set to True to remove the app

if UNINSTALL:
    run(f"helm -n {NAMESPACE} uninstall {RELEASE} --wait")
    run(f"kubectl -n {NAMESPACE} get all")

$ helm -n csc478 uninstall dashboard --wait
These resources were kept due to the resource policy:
[Secret] dashboard-db-credentials

release "dashboard" uninstalled

$ kubectl -n csc478 get all
No resources found in csc478 namespace.

